In [1]:
import cv2
import numpy as np
import os
from pathlib import Path
from aicsimageio import AICSImage
from tqdm import tqdm
from concurrent.futures import ProcessPoolExecutor, as_completed

# OpenCV 멀티스레딩 충돌 방지
cv2.setNumThreads(0)

def convert_single_image(input_path: Path, output_path: Path):
    """단일 파일을 변환하고 저장하는 핵심 로직"""
    try:
        # 저장할 대상의 부모 폴더가 없으면 생성 (경로 구조 유지의 핵심)
        output_path.parent.mkdir(parents=True, exist_ok=True)
        
        # 이미지 로드
        img = AICSImage(str(input_path))
        img_data = img.get_image_data("YX", T=0, C=0, Z=0)
        
        # 8-bit 정규화
        if img_data.dtype != np.uint8:
            img_norm = cv2.normalize(img_data, None, 0, 255, cv2.NORM_MINMAX)
            img_8bit = np.uint8(img_norm)
        else:
            img_8bit = img_data
            
        # PNG 저장
        cv2.imwrite(str(output_path), img_8bit)
        return True
        
    except Exception as e:
        print(f"\nError processing {input_path.name}: {e}")
        return False

def batch_convert_tiff_to_png(input_dir: str, output_dir: str):
    """지정된 최상단 폴더 아래의 모든 TIFF를 동일한 구조로 변환하는 함수"""
    in_root = Path(input_dir)
    out_root = Path(output_dir)
    
    # 1. 입력 폴더 내의 모든 .tif 및 .tiff 파일 재귀적(rglob)으로 탐색
    tiff_files = [
        f for f in in_root.rglob("*") 
        if f.suffix.lower() in ['.tif', '.tiff']
    ]
    
    print(f"총 {len(tiff_files)}개의 TIFF 파일을 찾았습니다.")
    if not tiff_files:
        return
        
    # 2. (입력 파일 경로, 출력 파일 경로) 쌍을 담을 리스트 생성
    conversion_tasks = []
    for file_path in tiff_files:
        # 최상단 폴더(in_root)를 떼어내고 나머지 하위 경로만 추출
        # 예: breast/bt549/nybd1/image.tiff
        rel_path = file_path.relative_to(in_root)
        
        # 출력 최상단 폴더(out_root)에 하위 경로를 붙이고, 확장자만 .png로 변경
        target_path = out_root / rel_path.with_suffix('.png')
        conversion_tasks.append((file_path, target_path))
        
    # 3. 서버 CPU 코어를 모두 활용한 병렬 처리
    max_cores = os.cpu_count()
    print(f"🚀 {max_cores}개의 CPU 코어를 사용하여 병렬 변환을 시작합니다...")
    
    # ProcessPoolExecutor를 이용해 고속 변환
    with ProcessPoolExecutor(max_workers=max_cores) as executor:
        # executor.submit을 사용해 작업 예약
        futures = [executor.submit(convert_single_image, src, dst) for src, dst in conversion_tasks]
        
        # tqdm으로 전체 진행률 모니터링
        for future in tqdm(as_completed(futures), total=len(futures), desc="Converting"):
            try:
                future.result()
            except Exception as e:
                print(f"error arraising {e}")

    print("\n✅ 모든 변환 및 구조 복사 작업이 완료되었습니다!")

# ==========================================
# 실행 부분
# ==========================================
if __name__ == "__main__":
    # 이 두 폴더를 최상단(root)으로 간주하고, 그 하위의 장기/세포주/화합물 폴더 구조를 그대로 복제합니다.
    INPUT_ROOT = "../data/tiff_files/NCI60 Images"
    OUTPUT_ROOT = "../data/png_converted/NCI60 Images"
    
    batch_convert_tiff_to_png(INPUT_ROOT, OUTPUT_ROOT)

총 34696개의 TIFF 파일을 찾았습니다.
🚀 64개의 CPU 코어를 사용하여 병렬 변환을 시작합니다...


Converting: 100%|██████████| 34696/34696 [04:07<00:00, 140.32it/s]



✅ 모든 변환 및 구조 복사 작업이 완료되었습니다!
